In [ ]:
# Bootstrap: make src/ importable from notebooks in src/shared/
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

# Baseline Pipeline

End-to-end ERM baseline: frozen DINOv2 (ViT-S/14, 384-d CLS token) → Encoder MLP (384 → 128 → 32) → Head MLP (32 → 32 → 5 biomass targets).
Cached DINOv2 features are reused when available; extraction runs automatically when the cache is missing.
Set `SKIP_TRAINING = True` to reload a saved checkpoint instead of re-training.

## 1. Imports and Configuration

Load all dependencies and define training hyperparameters and path constants.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from shared.data_utils import load_datasets
from shared.model import BiomassModel
from shared.metrics import TARGETS
from shared.train import train, eval_epoch

REPO_ROOT = Path("../..")
CACHE_DIR  = REPO_ROOT / "src" / "cache"
CKPT_DIR   = REPO_ROOT / "src" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

cfg = dict(
    seed         = 42,
    epochs       = 80,
    lr           = 3e-4,
    weight_decay = 1e-3,
    latent_dim   = 32,
    dropout      = 0.1,
    batch_size   = 32,
)

SKIP_TRAINING = False  # set True to reload baseline.pt instead of re-training

device = (
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()          else "cpu"
)
print(f"Device : {device}")
print("Config :", cfg)

## 2. Feature Extraction

Check whether the DINOv2 feature cache exists; run extraction on real training and test images only if the cache files are missing.

In [ ]:
from shared.extract_features import (
    DINOV2_CACHE, IDS_CACHE,
    TEST_IMAGE_CACHE, TEST_IDS_CACHE,
    extract_dinov2,
)

IMAGE_DIR      = REPO_ROOT / "data" / "image" / "train"
TEST_IMAGE_DIR = REPO_ROOT / "data" / "image" / "test"

train_cached = DINOV2_CACHE.exists() and IDS_CACHE.exists()
test_cached  = TEST_IMAGE_CACHE.exists() and TEST_IDS_CACHE.exists()

if train_cached and test_cached:
    print("Cache found — skipping extraction.")
    print(f"  Train features : {np.load(DINOV2_CACHE).shape}")
    print(f"  Test  features : {np.load(TEST_IMAGE_CACHE).shape}")
else:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    if not train_cached:
        train_paths = sorted(IMAGE_DIR.glob("*.jpg"))
        print(f"Extracting DINOv2 features for {len(train_paths)} training images...")
        feats = extract_dinov2(train_paths, device)
        ids   = np.array([p.stem for p in train_paths])
        np.save(DINOV2_CACHE, feats)
        np.save(IDS_CACHE, ids)
        print(f"  Saved: {DINOV2_CACHE}  shape={feats.shape}")

    if not test_cached:
        test_paths = sorted(TEST_IMAGE_DIR.glob("*.jpg"))
        print(f"Extracting DINOv2 features for {len(test_paths)} test images...")
        test_feats = extract_dinov2(test_paths, device)
        test_ids   = np.array([p.stem for p in test_paths])
        np.save(TEST_IMAGE_CACHE, test_feats)
        np.save(TEST_IDS_CACHE, test_ids)
        print(f"  Saved: {TEST_IMAGE_CACHE}  shape={test_feats.shape}")

## 3. Data Loading

Load cached features from disk and assemble GroupKFold train / validation PyTorch datasets.

In [ ]:
CSV_PATH = REPO_ROOT / "data" / "tabular" / "train" / "train.csv"

# load_datasets builds BiomassFeatureDataset pairs using the GroupKFold fold-0 split
train_ds, val_ds = load_datasets(CSV_PATH, CACHE_DIR)

train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

print(f"Train samples : {len(train_ds)}")
print(f"Val   samples : {len(val_ds)}")
print(f"Feature dim   : {train_ds[0][0].shape[0]}")

## 4. Model Initialization

Instantiate `BiomassModel`: the Encoder compresses 384-d DINOv2 features to a 32-d latent, then the Head maps the latent to the 5 biomass targets.

In [ ]:
torch.manual_seed(cfg["seed"])

# BiomassModel = Encoder(384 -> 128 -> 32) + Head(32 -> 32 -> 5)
model = BiomassModel(
    input_dim  = 384,
    latent_dim = cfg["latent_dim"],
    output_dim = 5,
    dropout    = cfg["dropout"],
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters : {n_params:,}")
print(model)

## 5. Training

Train with AdamW and cosine-annealing LR using a weighted SmoothL1 loss; save the checkpoint on completion, or reload it when `SKIP_TRAINING = True`.

In [ ]:
ckpt_path = CKPT_DIR / "baseline.pt"

if SKIP_TRAINING and ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    history = ckpt.get("history", {})
    print(f"Loaded checkpoint from {ckpt_path}")
else:
    history = train(
        model, train_ds, val_ds,
        epochs       = cfg["epochs"],
        batch_size   = cfg["batch_size"],
        lr           = cfg["lr"],
        weight_decay = cfg["weight_decay"],
        seed         = cfg["seed"],
        device       = device,
        verbose      = True,
    )
    torch.save({"model_state_dict": model.state_dict(), "history": history}, ckpt_path)
    print(f"\nCheckpoint saved: {ckpt_path}")

## 6. Validation Results

Compute the weighted global R^2 (the Kaggle competition metric) and per-target RMSE on the held-out validation fold.

In [ ]:
val_loss, val_r2, val_rmse, y_pred, y_true = eval_epoch(model, val_loader, device)

print(f"Validation loss : {val_loss:.4f}")
print(f"Weighted R2     : {val_r2:.4f}")
print()
print(f"{'Target':<20}  RMSE")
print("-" * 32)
for t in TARGETS:
    print(f"  {t:<18}  {val_rmse[t]:.4f}")

## 7. Training Curves

Visualise train/val loss and validation R^2 over epochs to check for overfitting or underfitting.

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(epochs_range, history["train_loss"], label="Train loss", color="steelblue")
ax.plot(epochs_range, history["val_loss"],   label="Val loss",   color="steelblue", linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Weighted SmoothL1 loss")
ax.set_title("Training and Validation Loss")
ax.legend()

ax = axes[1]
ax.plot(epochs_range, history["val_r2"], label="Val R2", color="darkorange")
ax.axhline(val_r2, color="grey", linestyle=":", label=f"Final R2={val_r2:.4f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Weighted global R2")
ax.set_title("Validation R2")
ax.legend()

plt.tight_layout()
plt.show()

## 8. Kaggle Submission (Optional)

Generate a long-format submission CSV from test-set predictions; runs only when the test feature cache is present.

In [ ]:
import pandas as pd

if not TEST_IMAGE_CACHE.exists():
    print("Test features not cached — re-run Cell 2 (Feature Extraction) first.")
else:
    test_feats = np.load(TEST_IMAGE_CACHE).astype(np.float32)
    test_ids   = np.load(TEST_IDS_CACHE,  allow_pickle=True)

    model.eval()
    with torch.no_grad():
        preds = model(
            torch.tensor(test_feats, dtype=torch.float32, device=device)
        ).cpu().numpy()

    records = [
        {"sample_id": f"{img_id}__{target}", "target": float(pred_row[t_idx])}
        for img_id, pred_row in zip(test_ids, preds)
        for t_idx, target in enumerate(TARGETS)
    ]
    sub_df = pd.DataFrame(records)

    sub_path = REPO_ROOT / "src" / "shared" / "submission_baseline.csv"
    sub_df.to_csv(sub_path, index=False)
    print(f"Submission saved: {sub_path}  ({len(sub_df)} rows)")
    sub_df.head()